In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp02
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import logging
import os
from dataclasses import dataclass, field, replace
from pathlib import Path
from typing import Any, Literal

import numpy as np
import polars as pl
from dotenv import load_dotenv

logger = logging.getLogger(__name__)


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ════════════════════════════════════════════════════════════════════════
# SPLIT FIRST, THEN PREPROCESS
# ════════════════════════════════════════════════════════════════════════
# kailash-ml's ``PreprocessingPipeline.setup()`` fits its imputer, its
# categorical encoder (including TARGET encoding) and its scaler on EVERY
# row it is given, and only afterwards splits train/test. Handing it the
# whole dataset therefore lets test-set statistics leak into training.
#
# The helpers below hold the test rows out FIRST, fit the pipeline on the
# training rows only, and push the held-out rows through the fitted
# ``transform()``. ``setup()`` still splits internally, but that split now
# happens inside the training rows, so it cannot leak anything.

_SETUP_OPTIONS: frozenset[str] = frozenset(
    {
        "normalize",
        "normalize_method",
        "categorical_encoding",
        "imputation_strategy",
        "impute_n_neighbors",
        "remove_outliers",
        "outlier_threshold",
        "max_cardinality",
        "exclude_columns",
        "remove_multicollinearity",
        "multicollinearity_threshold",
        "pca",
        "pca_components",
        "transform_target",
        "fix_imbalance",
        "imbalance_method",
    }
)

# Arguments the helpers own themselves; passing them through would
# re-introduce the leak (``data`` = the full frame) or fight the split.
_OWNED_BY_HELPER: frozenset[str] = frozenset({"data", "target", "train_size", "seed"})


@dataclass(frozen=True)
class PreprocessResult:
    """Outcome of :func:`split_then_preprocess` / :func:`preprocess_train_test`.

    Mirrors the fields of kailash-ml's ``SetupResult`` that exercises use,
    plus the fitted pipeline and a record of what was done to which split.

    ``train_data`` / ``test_data``
        Both produced by the SAME pipeline, fitted on the training rows only.
    ``transformers``
        The fitted imputer / encoder / scaler state (e.g.
        ``transformers["ordinal_mappings"]``), learned from training rows.
    ``train_only_steps``
        Steps applied to the training split only (outlier removal,
        class-imbalance resampling). The test split is never filtered or
        resampled — it stays a faithful sample of the data.
    ``test_null_fills``
        Columns that were complete in the training rows but have gaps in
        the test rows (the fitted pipeline has no learned fill for them),
        mapped to the TRAINING statistic used to fill them.
    """

    train_data: pl.DataFrame
    test_data: pl.DataFrame
    target_column: str
    task_type: str
    numeric_columns: list[str]
    categorical_columns: list[str]
    transformers: dict[str, Any]
    pipeline: Any
    original_shape: tuple[int, int]
    train_raw_rows: int
    test_raw_rows: int
    stratified: bool
    train_only_steps: tuple[str, ...] = ()
    test_null_fills: dict[str, Any] = field(default_factory=dict)
    summary: str = ""


def _looks_like_classification(series: pl.Series) -> bool:
    """Same rule kailash-ml uses to call a target "classification"."""
    if series.dtype in (pl.Boolean, pl.Categorical, pl.Utf8, pl.String):
        return True
    return series.drop_nulls().n_unique() <= 20


def split_raw_train_test(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
) -> tuple[pl.DataFrame, pl.DataFrame, bool]:
    """Hold out test rows from the RAW frame before anything is fitted.

    Rows are shuffled with ``seed``. ``test_size`` is a fraction in (0, 1)
    or an exact row count. With ``stratify="auto"`` a classification target
    (same rule as kailash-ml) is stratified so every class keeps its share
    in both splits; ``True`` / ``False`` force the choice.

    Returns ``(train_raw, test_raw, stratified)``; both splits keep the
    shuffled order.
    """
    if target not in df.columns:
        raise ValueError(f"Target column '{target}' not found in data.")
    n = df.height
    if n < 2:
        raise ValueError(f"Need at least 2 rows to split, got {n}.")
    if isinstance(test_size, np.integer):
        test_size = int(test_size)
    if isinstance(test_size, bool) or not isinstance(test_size, (int, float)):
        raise TypeError(
            f"test_size must be a fraction in (0, 1) or a row count, got {test_size!r}."
        )
    if isinstance(test_size, int):
        if not 1 <= test_size < n:
            raise ValueError(f"test_size as a row count must be in [1, {n - 1}], got {test_size}.")
        test_fraction = test_size / n
    else:
        if not 0.0 < test_size < 1.0:
            raise ValueError(f"test_size as a fraction must be in (0, 1), got {test_size}.")
        test_fraction = float(test_size)

    is_classification = _looks_like_classification(df[target])
    if stratify == "auto":
        stratified = is_classification
    elif stratify is True:
        if not is_classification:
            raise ValueError(
                f"stratify=True needs a classification target; '{target}' has "
                f"{df[target].drop_nulls().n_unique()} distinct values."
            )
        stratified = True
    elif stratify is False:
        stratified = False
    else:
        raise TypeError(f"stratify must be True, False or 'auto', got {stratify!r}.")

    perm = np.random.RandomState(seed).permutation(n)
    shuffled = df[perm.tolist()]

    if stratified:
        # Within each class (in shuffled order), the first round(n_c * f)
        # rows go to test — the class share is the same in both splits.
        is_test = pl.int_range(pl.len()).over(target) < (
            pl.len().over(target) * test_fraction
        ).round()
        flagged = shuffled.with_columns(is_test.alias("__split_is_test"))
        test_raw = flagged.filter(pl.col("__split_is_test")).drop("__split_is_test")
        train_raw = flagged.filter(~pl.col("__split_is_test")).drop("__split_is_test")
    else:
        n_test = test_size if isinstance(test_size, int) else int(round(n * test_fraction))
        train_raw = shuffled.head(n - n_test)
        test_raw = shuffled.tail(n_test)

    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError(
            f"Split produced an empty side (train={train_raw.height}, "
            f"test={test_raw.height}); adjust test_size."
        )
    return train_raw, test_raw, stratified


def _fill_unfitted_test_nulls(
    test_raw: pl.DataFrame,
    train_raw: pl.DataFrame,
    fitted: Any,
    strategy: str,
) -> tuple[pl.DataFrame, dict[str, Any]]:
    """Fill test gaps the fitted pipeline has no learned value for.

    kailash-ml only stores a fill value for columns that had gaps in the
    rows it was fitted on. A column that is complete in the training rows
    but has gaps in the test rows would otherwise reach the model as null.
    The fill is the TRAINING statistic for the same strategy (never a test
    statistic), and every such column is logged and returned.
    """
    if strategy == "drop":
        return test_raw, {}  # transform() drops incomplete rows itself
    learned = set(fitted.transformers.get("imputer_stats", {}))
    if strategy in ("knn", "iterative"):
        learned |= set(fitted.transformers.get("sklearn_imputer_cols", []))

    fills: dict[str, Any] = {}
    for col in fitted.numeric_columns + fitted.categorical_columns:
        if col in learned or col not in test_raw.columns:
            continue
        if test_raw[col].null_count() == 0:
            continue
        train_col = train_raw[col].drop_nulls()
        if col in fitted.categorical_columns or strategy == "mode":
            modes = train_col.mode().sort()
            if len(modes) == 0:
                raise ValueError(f"Cannot fill '{col}': no observed training values.")
            value = modes[0]
        elif strategy == "median":
            value = train_col.median()
        else:  # mean (and the numeric fallback for any other strategy)
            value = train_col.mean()
        if value is None:
            raise ValueError(f"Cannot fill '{col}': no observed training values.")
        fills[col] = value
    if fills:
        logger.warning(
            "Test rows have gaps in columns that were complete in training; "
            "filled with TRAINING statistics: %s",
            fills,
        )
        test_raw = test_raw.with_columns(
            [pl.col(c).fill_null(v) for c, v in fills.items()]
        )
    return test_raw, fills


def preprocess_train_test(
    train_raw: pl.DataFrame,
    test_raw: pl.DataFrame,
    target: str,
    *,
    seed: int = 42,
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Fit ``PreprocessingPipeline`` on ``train_raw`` only; transform both.

    ``setup_kwargs`` are ``PreprocessingPipeline.setup()`` options
    (``normalize``, ``categorical_encoding``, ``imputation_strategy`` …).
    Steps that change the ROWS of a split are applied to training rows only:

    * ``remove_outliers`` — outlier rows are dropped from training only.
    * ``fix_imbalance`` — SMOTE / ADASYN (or the class-weight flag) is
      applied to the whole transformed training split, never to test.

    Use this directly inside cross-validation folds; use
    :func:`split_then_preprocess` for a single train/test split.
    """
    from kailash_ml import PreprocessingPipeline

    owned = sorted(_OWNED_BY_HELPER & set(setup_kwargs))
    if owned:
        raise TypeError(
            f"{owned} are set by the split-first helper itself and cannot be "
            "passed through to setup()."
        )
    unknown = sorted(set(setup_kwargs) - _SETUP_OPTIONS)
    if unknown:
        raise TypeError(f"Unknown PreprocessingPipeline.setup() options: {unknown}.")
    for name, frame in (("train_raw", train_raw), ("test_raw", test_raw)):
        if target not in frame.columns:
            raise ValueError(f"Target column '{target}' not found in {name}.")
    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError("train_raw and test_raw must both have rows.")
    if set(train_raw.columns) != set(test_raw.columns):
        raise ValueError("train_raw and test_raw must have the same columns.")

    fix_imbalance = bool(setup_kwargs.pop("fix_imbalance", False))
    imbalance_method = setup_kwargs.get("imbalance_method", "smote")
    remove_outliers = bool(setup_kwargs.get("remove_outliers", False))

    pipeline = PreprocessingPipeline()
    # The pipeline only ever sees training rows. Its internal split stays
    # inside them; resampling is applied below to the full training split.
    fitted = pipeline.setup(
        data=train_raw, target=target, seed=seed, fix_imbalance=False, **setup_kwargs
    )

    train_only: list[str] = []
    if remove_outliers:
        # setup() removed the outlier rows (training rows only); its two
        # internal pieces together are every surviving training row.
        train_data = pl.concat([fitted.train_data, fitted.test_data], how="vertical")
        train_only.append(
            f"remove_outliers: {train_raw.height - train_data.height} training rows dropped"
        )
    else:
        train_data = pipeline.transform(train_raw)

    strategy = setup_kwargs.get("imputation_strategy", "mean")
    test_filled, test_fills = _fill_unfitted_test_nulls(test_raw, train_raw, fitted, strategy)
    test_data = pipeline.transform(test_filled)

    if fix_imbalance and fitted.task_type == "classification":
        resample = getattr(pipeline, "_apply_imbalance_correction", None)
        if resample is None:
            raise RuntimeError(
                "This kailash-ml version has no imbalance-correction step on "
                "PreprocessingPipeline; cannot apply fix_imbalance to the training split."
            )
        before = train_data.height
        train_data = resample(train_data, target, imbalance_method, seed)
        train_only.append(
            f"fix_imbalance ({imbalance_method}): training rows {before} -> {train_data.height}"
        )

    summary = "\n".join(
        [
            f"Task type: {fitted.task_type}",
            f"Split first: {train_raw.height} train rows / {test_raw.height} test rows "
            "(pipeline fitted on the train rows only)",
            f"Numeric features: {len(fitted.numeric_columns)}",
            f"Categorical features: {len(fitted.categorical_columns)}",
            f"Encoding: {setup_kwargs.get('categorical_encoding', 'onehot')}",
            f"Normalization: {'yes' if setup_kwargs.get('normalize', True) else 'no'}",
            f"Imputation: {strategy}",
            f"Train-only steps: {'; '.join(train_only) if train_only else 'none'}",
            f"Transformed train: {train_data.height} rows x {train_data.width} cols",
            f"Transformed test: {test_data.height} rows x {test_data.width} cols",
        ]
    )

    return PreprocessResult(
        train_data=train_data,
        test_data=test_data,
        target_column=target,
        task_type=fitted.task_type,
        numeric_columns=list(fitted.numeric_columns),
        categorical_columns=list(fitted.categorical_columns),
        transformers=dict(fitted.transformers),
        pipeline=pipeline,
        original_shape=(train_raw.height + test_raw.height, train_raw.width),
        train_raw_rows=train_raw.height,
        test_raw_rows=test_raw.height,
        stratified=False,
        train_only_steps=tuple(train_only),
        test_null_fills=test_fills,
        summary=summary,
    )


def split_then_preprocess(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Leak-free replacement for ``PreprocessingPipeline().setup(df, ...)``.

    1. Split the RAW frame first (seeded shuffle; stratified on the target
       for classification when ``stratify="auto"``).
    2. Fit ``PreprocessingPipeline`` on the training rows only.
    3. ``transform()`` the test rows with the fitted pipeline.

    ``setup_kwargs`` are passed to ``PreprocessingPipeline.setup()``; see
    :func:`preprocess_train_test` for how row-changing options are handled.
    """
    train_raw, test_raw, stratified = split_raw_train_test(
        df, target, test_size=test_size, seed=seed, stratify=stratify
    )
    result = preprocess_train_test(train_raw, test_raw, target, seed=seed, **setup_kwargs)
    split_line = "stratified on the target" if stratified else "random"
    return replace(
        result,
        stratified=stratified,
        summary=result.summary.replace("Split first:", f"Split first ({split_line}):", 1),
    )


# ── shared/data_loader.py ──
"""Unified data loading for MLFP course — supports local and Colab."""


import logging
from pathlib import Path

import polars as pl

logger = logging.getLogger(__name__)

# Google Drive shared folder containing all MLFP datasets
_DRIVE_FOLDER_ID = "16c3RkGmiwMWbjD7cJKbJx-JRZlgmQdws"

# Module subfolders on the shared Drive
_MODULES = {
    "mlfp01",
    "mlfp02",
    "mlfp03",
    "mlfp04",
    "mlfp05",
    "mlfp06",
    "mlfp_assessment",
}


def _is_colab() -> bool:
    """Detect if running inside Google Colab."""
    try:
        import google.colab  # noqa: F401

        return True
    except ImportError:
        return False


def _colab_data_root() -> Path:
    """Return the Drive-mounted mlfp_data path in Colab."""
    return Path("/content/drive/MyDrive/mlfp_data")


def _local_cache_dir() -> Path:
    """Return local cache directory for downloaded files."""
    cache = Path.cwd() / ".data_cache"
    cache.mkdir(exist_ok=True)
    return cache


def _download_from_drive(module: str, filename: str, dest: Path) -> Path:
    """Download a file from the shared Google Drive using gdown."""
    import gdown

    dest_dir = dest / module
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest_file = dest_dir / filename

    if dest_file.exists():
        logger.debug("Using cached file: %s", dest_file)
        return dest_file

    # gdown can download from a folder by file path
    url = f"https://drive.google.com/drive/folders/{_DRIVE_FOLDER_ID}"
    logger.info("Downloading %s/%s from Google Drive...", module, filename)

    # Download the specific file from the shared folder
    try:
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
            remaining_ok=True,
        )
    except TypeError:
        # Older gdown versions don't support remaining_ok
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
        )

    if not dest_file.exists():
        # Try direct download if folder download didn't isolate the file
        for candidate in dest.rglob(filename):
            if candidate.is_file():
                if candidate != dest_file:
                    candidate.rename(dest_file)
                return dest_file

        msg = (
            f"File not found after download: {module}/{filename}. "
            f"Check that it exists in the mlfp_data shared Drive."
        )
        raise FileNotFoundError(msg)

    return dest_file


def _read_file(path: Path) -> pl.DataFrame:
    """Read a data file into a polars DataFrame."""
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pl.read_csv(path, try_parse_dates=True)
    elif suffix == ".parquet":
        return pl.read_parquet(path)
    elif suffix == ".json":
        return pl.read_json(path)
    elif suffix in (".p", ".pickle", ".pkl"):
        import pickle

        with open(path, "rb") as f:
            obj = pickle.load(f)  # noqa: S301
        if isinstance(obj, pl.DataFrame):
            return obj
        raise TypeError(
            f"Cannot convert pickle object of type {type(obj)} to polars DataFrame. "
            f"Convert the pickle to parquet upstream: pl.from_pandas(obj).write_parquet('out.parquet')"
        )
    else:
        raise ValueError(
            f"Unsupported file format: {suffix}. Use .csv, .parquet, or .json"
        )


def _repo_data_dir() -> Path | None:
    """Find the repo-local data/ directory by walking up from cwd."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data"
        if candidate.is_dir() and (parent / "pyproject.toml").exists():
            return candidate
    return None


class MLFPDataLoader:
    """Load MLFP course datasets with automatic source resolution.

    Resolution order:
    1. Colab: Drive mount at /content/drive/MyDrive/mlfp_data/
    2. Local repo data/ directory (committed datasets)
    3. Google Drive download via gdown (cached in .data_cache/)

    Usage:
        loader = MLFPDataLoader()
        df = loader.load("mlfp01", "hdbprices.csv")

    Shortcut:
        df = MLFPDataLoader.mlfp01("hdbprices.csv")
    """

    def __init__(self, cache_dir: Path | str | None = None):
        self._colab = _is_colab()
        if self._colab:
            self._root = _colab_data_root()
        else:
            self._local_data = _repo_data_dir()
            self._cache = Path(cache_dir) if cache_dir else _local_cache_dir()

    def load_raw(self, module: str, filename: str) -> Path:
        """Return the file path without reading into memory.

        Use this for image directories, audio files, or any data that torch/HF
        loads directly rather than via polars.

        Args:
            module: Module subfolder (e.g., "mlfp05")
            filename: File or directory name (e.g., "fashion_mnist", "cifar10")

        Returns:
            Path to the local file or directory.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
        else:
            if self._local_data:
                local_path = self._local_data / module / filename
                if local_path.exists():
                    return local_path
            path = self._cache / module / filename

        if not path.exists():
            raise FileNotFoundError(
                f"Raw data not found: {module}/{filename}. "
                f"Run 'python scripts/fetch-real-data.py' to download."
            )
        return path

    @staticmethod
    def load_hf(
        dataset_name: str,
        split: str = "train",
        streaming: bool = False,
    ):
        """Load a HuggingFace dataset directly (not via polars).

        Use this for large datasets (millions of rows) or multimodal data
        (images, audio) that don't fit into a DataFrame.

        Args:
            dataset_name: HuggingFace dataset ID (e.g., "zalando-datasets/fashion_mnist")
            split: Dataset split ("train", "test", "validation")
            streaming: If True, returns an IterableDataset for memory-efficient processing

        Returns:
            HuggingFace Dataset or IterableDataset object.
        """
        from datasets import load_dataset

        logger.info(
            "Loading HuggingFace dataset: %s (split=%s, streaming=%s)",
            dataset_name,
            split,
            streaming,
        )
        return load_dataset(dataset_name, split=split, streaming=streaming)

    def load(self, module: str, filename: str) -> pl.DataFrame:
        """Load a dataset file as a polars DataFrame.

        Args:
            module: Module subfolder (e.g., "mlfp01", "mlfp_assessment")
            filename: File name within the module folder (e.g., "hdbprices.csv")

        Returns:
            polars DataFrame with the loaded data.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
            if not path.exists():
                raise FileNotFoundError(
                    f"File not found: {path}. "
                    f"Ensure mlfp_data is accessible in your Google Drive."
                )
        else:
            # Check repo-local data/ first, then fall back to Drive download
            if self._local_data:
                module_dir = self._local_data / module
                local_path = module_dir / filename
                if local_path.exists():
                    path = local_path
                    logger.info(
                        "Loading %s/%s from local data/ (%s)", module, filename, path
                    )
                    return _read_file(path)
                # Bundled data is present for this module but the file is absent
                # → it is genuinely missing. Raise immediately instead of falling
                # back to a Drive download that would hang offline (and pulls the
                # whole folder online just to discover the file is not there).
                if module_dir.is_dir():
                    raise FileNotFoundError(
                        f"File not found: {module}/{filename} "
                        f"(not in local data/{module}/)."
                    )
            path = _download_from_drive(module, filename, self._cache)

        logger.info("Loading %s/%s (%s)", module, filename, path)
        return _read_file(path)

    def list_files(self, module: str) -> list[str]:
        """List available data files in a module folder."""
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            root = self._root / module
        else:
            root = self._cache / module

        if not root.exists():
            return []

        return sorted(f.name for f in root.iterdir() if f.is_file())

    # -- Module shortcuts --

    @classmethod
    def mlfp01(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp01 (Data Pipelines & Visualisation)."""
        return cls().load("mlfp01", filename)

    @classmethod
    def mlfp02(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp02 (Statistical Mastery)."""
        return cls().load("mlfp02", filename)

    @classmethod
    def mlfp03(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp03 (Supervised ML)."""
        return cls().load("mlfp03", filename)

    @classmethod
    def mlfp04(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp04 (Unsupervised ML)."""
        return cls().load("mlfp04", filename)

    @classmethod
    def mlfp05(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp05 (Deep Learning & Vision)."""
        return cls().load("mlfp05", filename)

    @classmethod
    def mlfp06(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp06 (LLMs, Agents & Transformation)."""
        return cls().load("mlfp06", filename)

    @classmethod
    def assessment(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp_assessment (capstone datasets)."""
        return cls().load("mlfp_assessment", filename)


# ── shared/mlfp02/ex_4.py ──
"""
Shared infrastructure for MLFP02 Exercise 4 — A/B Testing & Experiment Design.

Contains: experiment data loading, two-arm extraction, power-analysis
primitives (z-values, required-n helper), random number generator factory,
and a small print helper used across the four technique files.

Technique-specific logic (power curves, SRM detection, Welch's t-test,
adaptive design) lives in the per-technique files — not here.
"""

import math
from pathlib import Path
from typing import NamedTuple

import numpy as np
import polars as pl
from scipy import stats


# ════════════════════════════════════════════════════════════════════════
# DESIGN CONSTANTS
# ════════════════════════════════════════════════════════════════════════

ALPHA: float = 0.05
POWER_TARGET: float = 0.80
DESIGN_MDE_PCT: float = 2.0  # relative MDE (percent of baseline mean)
SEED: int = 42

# Output directory for plots and comparison tables
OUTPUT_DIR = Path("outputs") / "mlfp02_ex4_experiment_design"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# The experiment was DESIGNED with unequal allocation across four arms.
# Exercise 4 analyses control vs treatment_a, so its SRM check must use
# the designed control share within that pair (40 / (40 + 35)), never 50/50.
DESIGNED_ALLOCATION: dict[str, float] = {
    "control": 0.40,
    "treatment_a": 0.35,
    "treatment_b": 0.15,
    "variant_c": 0.10,
}
TREATMENT_ARM: str = "treatment_a"


# ════════════════════════════════════════════════════════════════════════
# DATA CONTAINER
# ════════════════════════════════════════════════════════════════════════


class TwoArmAB(NamedTuple):
    """Two-arm A/B subset pulled from the raw experiment frame."""

    experiment: pl.DataFrame  # full frame (all arms)
    ab_data: pl.DataFrame  # control + treatment_a only, sorted by `ts`
    ctrl_values: np.ndarray  # float64 numpy array, chronological order
    treat_values: np.ndarray  # float64 numpy array, chronological order
    n_control: int
    n_treatment: int
    n_total: int


# ════════════════════════════════════════════════════════════════════════
# DATA LOADING
# ════════════════════════════════════════════════════════════════════════


def load_experiment() -> TwoArmAB:
    """Load experiment_data.parquet and extract the two-arm A/B subset.

    Returns a TwoArmAB tuple with the raw frame, the filtered A/B frame,
    and numpy arrays for the control and treatment_a metric_value columns.
    The A/B frame gains a parsed datetime column `ts` and is sorted by it,
    so any "early vs late" comparison really is chronological (the raw
    file is NOT stored in time order).
    """
    loader = MLFPDataLoader()
    experiment = loader.load("mlfp02", "experiment_data.parquet")

    ab_data = (
        experiment.filter(
            pl.col("experiment_group").is_in(["control", TREATMENT_ARM])
        )
        .with_columns(pl.col("timestamp").str.to_datetime().alias("ts"))
        .sort("ts")
    )
    control = ab_data.filter(pl.col("experiment_group") == "control")
    treatment = ab_data.filter(pl.col("experiment_group") == TREATMENT_ARM)

    ctrl_values = control["metric_value"].to_numpy().astype(np.float64)
    treat_values = treatment["metric_value"].to_numpy().astype(np.float64)

    return TwoArmAB(
        experiment=experiment,
        ab_data=ab_data,
        ctrl_values=ctrl_values,
        treat_values=treat_values,
        n_control=control.height,
        n_treatment=treatment.height,
        n_total=ab_data.height,
    )


# ════════════════════════════════════════════════════════════════════════
# POWER-ANALYSIS PRIMITIVES
# ════════════════════════════════════════════════════════════════════════


def z_critical(
    alpha: float = ALPHA, power: float = POWER_TARGET
) -> tuple[float, float]:
    """Return (z_{alpha/2}, z_beta) — the two critical values used in sample-size formulas."""
    return stats.norm.ppf(1 - alpha / 2), stats.norm.ppf(power)


def required_n_per_group(
    sigma: float,
    mde_absolute: float,
    alpha: float = ALPHA,
    power: float = POWER_TARGET,
) -> int:
    """Two-sample normal-approximation sample size per group.

    n = (z_{alpha/2} + z_beta)^2 * 2 * sigma^2 / delta^2

    Used by both the up-front design phase (Exercise 4.1) and the
    adaptive/sequential re-estimation phase (Exercise 4.4).
    """
    if mde_absolute <= 0:
        raise ValueError("mde_absolute must be positive")
    z_alpha_half, z_beta = z_critical(alpha, power)
    return math.ceil((z_alpha_half + z_beta) ** 2 * 2 * sigma**2 / mde_absolute**2)


def power_at_n(
    n_per_group: int,
    sigma: float,
    mde_absolute: float,
    alpha: float = ALPHA,
) -> float:
    """Compute achieved power for a given per-group sample size.

    Uses the non-central normal approximation:
        power = 1 - Phi(z_{alpha/2} - ncp) + Phi(-z_{alpha/2} - ncp)
    where ncp = delta / (sigma * sqrt(2/n)).
    """
    z_alpha_half, _ = z_critical(alpha)
    se = sigma * np.sqrt(2 / n_per_group)
    ncp = mde_absolute / se
    return float(
        1 - stats.norm.cdf(z_alpha_half - ncp) + stats.norm.cdf(-z_alpha_half - ncp)
    )


def cohens_d(delta: float, sigma: float) -> float:
    """Cohen's d effect size for two-sample means with pooled sigma."""
    return float(delta / sigma)


def interpret_cohens_d(abs_d: float) -> str:
    """Cohen's conventions: <0.2 negligible, <0.5 small, <0.8 medium, else large."""
    if abs_d < 0.2:
        return "negligible"
    if abs_d < 0.5:
        return "small"
    if abs_d < 0.8:
        return "medium"
    return "large"


# ════════════════════════════════════════════════════════════════════════
# SAMPLE RATIO MISMATCH
# ════════════════════════════════════════════════════════════════════════


def designed_control_share(arm: str = TREATMENT_ARM) -> float:
    """Designed share of control within the (control, arm) pair."""
    c = DESIGNED_ALLOCATION["control"]
    return c / (c + DESIGNED_ALLOCATION[arm])


def srm_chisquare(
    n_control: int, n_treatment: int, expected_control_share: float
) -> tuple[float, float]:
    """χ² SRM test of an observed two-arm split against its DESIGNED split.

    Returns (chi2, p_value). Use p < 0.01 as the SRM alarm. There is no
    50/50 default on purpose: the expected split comes from the design.
    """
    n_total = n_control + n_treatment
    expected = np.array(
        [n_total * expected_control_share, n_total * (1 - expected_control_share)]
    )
    chi2, p = stats.chisquare(np.array([n_control, n_treatment]), f_exp=expected)
    return float(chi2), float(p)


# ════════════════════════════════════════════════════════════════════════
# RANDOM NUMBER GENERATOR
# ════════════════════════════════════════════════════════════════════════


def make_rng(seed: int = SEED) -> np.random.Generator:
    """Factory for the per-exercise numpy Generator — deterministic across runs."""
    return np.random.default_rng(seed)


# ════════════════════════════════════════════════════════════════════════
# REPORTING HELPERS
# ════════════════════════════════════════════════════════════════════════


def print_banner(title: str) -> None:
    """Print a uniform 70-char banner around a section title."""
    bar = "=" * 70
    print(f"\n{bar}\n  {title}\n{bar}")


def summarise_arm(name: str, values: np.ndarray) -> None:
    """Print a one-line summary of an experiment arm."""
    print(
        f"  {name:<10}: n={len(values):>6,}  "
        f"mean={values.mean():.4f}  std={values.std(ddof=1):.4f}"
    )




# ════════════════════════════════════════════════════════════════════════
# MLFP02 — Exercise 4.2: Sample Ratio Mismatch Detection
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Simulate an experiment with a known effect (positive control)
#   - Detect Sample Ratio Mismatch (SRM) via chi-squared test
#   - Simulate SRM to see how biased allocation inflates effect estimates
#   - Run SRM checks per segment, not just overall
#
# PREREQUISITES:
#   - MLFP02 Exercise 4.1 (experiment design, power analysis)
#   - MLFP02 Exercise 3 (chi-squared, hypothesis testing)
#
# ESTIMATED TIME: ~45 minutes
#
# TASKS (5-phase R10):
#   1. Theory — why SRM breaks experiments before they start
#   2. Build — positive control simulation with known treatment effect
#   3. Train — SRM detection on simulated and real data (vs the design)
#   4. Visualise — SRM impact: biased vs unbiased estimation
#   5. Apply — ride-hailing surge-pricing A/B with device-type SRM
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats



## THEORY — SRM Breaks Experiments Before They Start


In [ ]:
# Sample Ratio Mismatch (SRM) means the observed split between control
# and treatment differs from the designed split (usually 50/50).
#
# Common causes:
#   1. Bot filtering applied asymmetrically (bots blocked from treatment)
#   2. Redirect latency: treatment page loads slower, users bail
#   3. Randomisation bug: hash collision in user-ID bucketing
#   4. Post-randomisation filter: only "active" users analysed, but
#      treatment changes who looks "active"
#
# Why it matters:
#   SRM means the two groups are no longer comparable.  If high-value
#   users are more likely to end up in treatment, the treatment group
#   has a higher baseline regardless of the feature being tested.
#   The treatment effect estimate is BIASED.
#
# Detection:
#   chi-squared goodness-of-fit test comparing observed vs expected
#   group sizes.  Use a STRICT threshold (p < 0.01) because the cost
#   of missing SRM is much higher than the cost of investigating it.



## TASK 1 — LOAD data & prepare


In [ ]:
print_banner("Exercise 4.2 — SRM Detection")

data: TwoArmAB = load_experiment()
rng = make_rng(SEED)

summarise_arm("Control", data.ctrl_values)
summarise_arm("Treatment", data.treat_values)

sigma_pooled = data.ctrl_values.std(ddof=1)



## TASK 2 — BUILD: Positive Control Simulation


In [ ]:
# A positive control injects a KNOWN treatment effect into synthetic
# data.  If our analysis pipeline cannot detect the known effect,
# the pipeline has a bug.  This is the experiment-design equivalent
# of a unit test.

print_banner("Positive Control Simulation")

sim_n_per = 10_000
true_effect = 2.0
sim_mu_control = data.ctrl_values.mean()

sim_control = rng.normal(loc=sim_mu_control, scale=sigma_pooled, size=sim_n_per)
sim_treatment = rng.normal(
    loc=sim_mu_control + true_effect, scale=sigma_pooled, size=sim_n_per
)

print(f"True control mean:   {sim_mu_control:.4f}")
print(f"True treatment mean: {sim_mu_control + true_effect:.4f}")
print(f"True effect (delta): {true_effect:.4f}")
print(f"n per group:         {sim_n_per:,}")
print(f"\nRealised:")
print(f"  Control mean:   {sim_control.mean():.4f}")
print(f"  Treatment mean: {sim_treatment.mean():.4f}")
print(
    f"  Observed diff:  {sim_treatment.mean() - sim_control.mean():.4f} "
    f"(true: {true_effect})"
)

sim_t, sim_p = stats.ttest_ind(sim_treatment, sim_control, equal_var=False)
print(f"  t-statistic: {sim_t:.4f}, p-value: {sim_p:.2e}")
print(
    f"  {'DETECTED' if sim_p < ALPHA else 'MISSED'} "
    f"(positive control should be detected)"
)
# INTERPRETATION: If we fail to detect a true effect of 2.0 with
# n=10,000 per group, the analysis pipeline has a bug.



### Checkpoint 1


In [ ]:
assert sim_p < ALPHA, "Positive control must be detected"
assert (
    abs(sim_treatment.mean() - sim_control.mean() - true_effect) < 3.0
), "Realised effect should be within 3.0 of true"
print("\n>>> Checkpoint 1 passed — positive control validated\n")



## TASK 3 — TRAIN: SRM Detection (Simulated & Real)


In [ ]:
print_banner("SRM Detection — chi-squared test")

# --- On simulated data (designed 50/50 — should pass) ---
sim_obs = np.array([sim_n_per, sim_n_per])
sim_exp = np.array([sim_n_per, sim_n_per])
sim_chi2, sim_srm_p = stats.chisquare(sim_obs, f_exp=sim_exp)
print(f"\n--- Simulated (designed 50/50) ---")
print(
    f"chi2={sim_chi2:.4f}, p={sim_srm_p:.6f} -> "
    f"{'SRM' if sim_srm_p < 0.01 else 'OK'}"
)

# --- On real data: test against the DESIGNED split ---
# This experiment allocated 40% control / 35% treatment_a (plus two
# other arms), so within the analysed pair control SHOULD hold
# 40 / (40 + 35) = 53.3% of users — not 50%.
expected_share = designed_control_share()
real_obs = np.array([data.n_control, data.n_treatment])
real_exp = np.array(
    [data.n_total * expected_share, data.n_total * (1 - expected_share)]
)
chi2_stat, srm_p = stats.chisquare(real_obs, f_exp=real_exp)
print(f"\n--- Real Data (designed control share {expected_share:.3f}) ---")
print(
    f"Observed: {data.n_control:,} / {data.n_treatment:,} "
    f"({data.n_control / data.n_total:.4f}/{data.n_treatment / data.n_total:.4f})"
)
print(f"chi2={chi2_stat:.4f}, p={srm_p:.4f}")
if srm_p < 0.01:
    print("SRM DETECTED — investigate:")
    print("  1. Bot filtering differential")
    print("  2. Technical redirect issues")
    print("  3. Randomisation bug")
    print("  4. Post-randomisation population filter")
else:
    print("No SRM detected — split is consistent with design")

# The classic mistake: testing the same counts against 50/50
naive_chi2, naive_p = stats.chisquare(
    real_obs, f_exp=np.array([data.n_total / 2, data.n_total / 2])
)
print(
    f"\nSame counts tested against 50/50 (WRONG for this design): "
    f"chi2={naive_chi2:.1f}, p={naive_p:.2e}"
)
print("A false alarm like this trains teams to ignore SRM — always use the design.")



### Checkpoint 2


In [ ]:
assert sim_chi2 == 0.0, "Simulated equal groups should give chi2=0"
assert srm_p >= 0.01, "The real pair should match its DESIGNED split"
assert naive_p < 0.01, "Testing an unequal design against 50/50 raises a false alarm"
print("\n>>> Checkpoint 2 passed — SRM detection completed\n")



## TASK 3b — Simulating SRM Impact on Estimation


In [ ]:
# What happens when high-value users are systematically more likely
# to end up in treatment?  We run 1,000 simulated experiments with
# and without SRM and compare the estimation bias.

print_banner("Simulating SRM — Impact on Estimation")

n_srm_sim = 1000
srm_biases = []
no_srm_biases = []
true_lift = 1.0

for _ in range(n_srm_sim):
    # No SRM: random 50/50 allocation
    users = rng.normal(50, 10, size=2000)
    assigned = rng.choice([0, 1], size=2000)
    ctrl_outcome = users[assigned == 0] + rng.normal(0, 5, size=(assigned == 0).sum())
    treat_outcome = (
        users[assigned == 1] + true_lift + rng.normal(0, 5, size=(assigned == 1).sum())
    )
    no_srm_biases.append((treat_outcome.mean() - ctrl_outcome.mean()) - true_lift)

    # WITH SRM: high-value users 60% likely to end up in treatment
    assign_prob = np.where(users > 55, 0.6, 0.4)
    assigned_srm = (rng.random(size=2000) < assign_prob).astype(int)
    ctrl_out_srm = users[assigned_srm == 0] + rng.normal(
        0, 5, size=(assigned_srm == 0).sum()
    )
    treat_out_srm = (
        users[assigned_srm == 1]
        + true_lift
        + rng.normal(0, 5, size=(assigned_srm == 1).sum())
    )
    srm_biases.append((treat_out_srm.mean() - ctrl_out_srm.mean()) - true_lift)

print(f"True treatment effect: {true_lift}")
print(f"\nNo SRM:")
print(f"  Mean estimation bias: {np.mean(no_srm_biases):+.4f}")
print(f"  Std of bias: {np.std(no_srm_biases):.4f}")
print(f"\nWith SRM (high-value -> treatment):")
print(f"  Mean estimation bias: {np.mean(srm_biases):+.4f}")
print(f"  Std of bias: {np.std(srm_biases):.4f}")
bias_gap = np.mean(srm_biases) - np.mean(no_srm_biases)
print(f"\nSRM adds a POSITIVE bias of ~{bias_gap:+.2f}")
print("because high-value users inflate treatment outcomes.")
# INTERPRETATION: SRM doesn't just break the sample split — it
# introduces SYSTEMATIC BIAS in treatment effect estimates.  When
# high-value users are more likely to end up in treatment, the
# estimated lift is inflated.  This is why SRM detection is the
# first sanity check, not an optional nicety.



### Checkpoint 3


In [ ]:
assert abs(np.mean(no_srm_biases)) < 0.5, "No-SRM bias should be near zero"
assert abs(np.mean(srm_biases)) > abs(
    np.mean(no_srm_biases)
), "SRM should introduce larger bias"
print("\n>>> Checkpoint 3 passed — SRM impact demonstrated\n")



## TASK 4 — VISUALISE: SRM Bias Distributions


In [ ]:
fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=["No SRM (unbiased)", "With SRM (biased)"],
)
fig.add_trace(
    go.Histogram(x=no_srm_biases, nbinsx=40, name="No SRM", marker_color="#4CAF50"),
    row=1,
    col=1,
)
fig.add_trace(
    go.Histogram(x=srm_biases, nbinsx=40, name="SRM", marker_color="#F44336"),
    row=1,
    col=2,
)
fig.update_layout(
    title="SRM Impact on Treatment Effect Estimation Bias",
    height=400,
    template="plotly_white",
)
out_path = OUTPUT_DIR / "srm_simulation.html"
fig.write_html(str(out_path))
print(f"Saved: {out_path}")



### Checkpoint 4


In [ ]:
assert out_path.exists(), "SRM simulation plot must be saved"
print("\n>>> Checkpoint 4 passed — SRM visualisation saved\n")



## TASK 5 — APPLY: Ride-Hailing Surge Pricing A/B (illustrative)


In [ ]:
# A ride-hailing platform (illustrative numbers) tests a new surge-pricing
# algorithm with a designed 50/50 split. The feature reaches iOS first,
# so iOS riders are more likely to land in treatment (55%) and Android
# riders less likely (45%) — a device-type SRM. iOS riders also have
# higher average fares.

print_banner("Applied — Ride-Hailing Surge Pricing A/B")

n_rides = 20_000
ios_share = 0.45

ride_devices = rng.choice(
    ["iOS", "Android"],
    size=n_rides,
    p=[ios_share, 1 - ios_share],
)
# Biased assignment: iOS users 55% likely to get treatment, Android 45%
assign_prob_ride = np.where(ride_devices == "iOS", 0.55, 0.45)
ride_assignment = (rng.random(n_rides) < assign_prob_ride).astype(int)

# Fare depends on device (iOS riders take pricier trips)
base_fare = np.where(ride_devices == "iOS", 18.0, 12.0)
true_algo_effect = 0.5  # True effect is small: $0.50
ride_fare = (
    base_fare + ride_assignment * true_algo_effect + rng.normal(0, 4, size=n_rides)
)

# Aggregate SRM check
n_ctrl_ride = (ride_assignment == 0).sum()
n_treat_ride = (ride_assignment == 1).sum()
ride_obs = np.array([n_ctrl_ride, n_treat_ride])
ride_exp = np.array([n_rides / 2, n_rides / 2])
ride_chi2, ride_srm_p = stats.chisquare(ride_obs, f_exp=ride_exp)

# Per-segment SRM check — the design is 50/50 WITHIN every device too
segment_srm = {}
for device in ["iOS", "Android"]:
    in_seg = ride_devices == device
    n_c = int(((ride_assignment == 0) & in_seg).sum())
    n_t = int(((ride_assignment == 1) & in_seg).sum())
    seg_chi2, seg_p = stats.chisquare([n_c, n_t])
    segment_srm[device] = {"n_c": n_c, "n_t": n_t, "p": float(seg_p)}

naive_effect = (
    ride_fare[ride_assignment == 1].mean() - ride_fare[ride_assignment == 0].mean()
)
# Remedy for the estimate: compare within each device, then re-weight by
# device share (a stratified estimate)
stratified_effect = sum(
    (ride_devices == dev).mean()
    * (
        ride_fare[(ride_assignment == 1) & (ride_devices == dev)].mean()
        - ride_fare[(ride_assignment == 0) & (ride_devices == dev)].mean()
    )
    for dev in ["iOS", "Android"]
)

print(f"n = {n_rides:,}  |  True algo effect: ${true_algo_effect:.2f}")
print(f"Control: {n_ctrl_ride:,}  Treatment: {n_treat_ride:,}")
print(
    f"Aggregate SRM: chi2={ride_chi2:.2f}, p={ride_srm_p:.4f} -> "
    f"{'DETECTED' if ride_srm_p < 0.01 else 'not detected'}"
)
for device, r in segment_srm.items():
    print(
        f"  {device:<8} SRM: {r['n_c']:,} vs {r['n_t']:,}, p={r['p']:.2e} -> "
        f"{'DETECTED' if r['p'] < 0.01 else 'not detected'}"
    )
any_segment_srm = any(r["p"] < 0.01 for r in segment_srm.values())
if ride_srm_p >= 0.01 and any_segment_srm:
    print(
        "\nThe aggregate test MISSES it: iOS over-allocation and Android\n"
        "under-allocation largely cancel in the totals. Only the per-segment\n"
        "check reveals the compositional imbalance."
    )
print(f"\nNaive estimated effect:      ${naive_effect:.2f}")
print(f"Stratified (within-device):  ${stratified_effect:.2f}")
print(f"True effect:                 ${true_algo_effect:.2f}")
print(f"Bias of the naive estimate:  ${naive_effect - true_algo_effect:+.2f}")
# INTERPRETATION: Device-type SRM is common in mobile A/B tests because
# iOS and Android releases ship at different times. Always run SRM checks
# per key segment (device, country, new vs returning), not just overall.
# Stratifying the estimate removes the composition bias here, but it
# cannot fix unobserved differences — investigate the cause before
# trusting any result.



### Checkpoint 5


In [ ]:
assert n_rides == n_ctrl_ride + n_treat_ride, "All users accounted for"
assert any_segment_srm, "The per-segment check should catch the device SRM"
assert abs(stratified_effect - true_algo_effect) < abs(
    naive_effect - true_algo_effect
), "Stratifying by device should reduce the bias"
print("\n>>> Checkpoint 5 passed — device-type SRM scenario completed\n")



## REFLECTION


- Positive controls: inject known effect to validate pipeline
  - SRM detection: chi-squared test on observed vs expected split
  - SRM causes: bot filtering, redirects, randomisation bugs
  - SRM impact: biased allocation => biased treatment effect estimates
  - SRM must be tested against the DESIGNED split, overall AND per segment
  - Applied: ride-hailing device-type SRM and a stratified estimate

  NEXT: In Exercise 4.3 you'll learn Welch's t-test — the robust
  test for comparing means when variances may differ.


In [ ]:
print("=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

print(">>> Exercise 4.2 complete — SRM Detection")

